# Analyze Project Document

## Overall Goals
The project involves the Khana dataset for Indian cuisine classification with 80 classes and 131,000+ images. The main tasks are:
1. Image Classification: Train a classifier to achieve >91% validation accuracy on 80 classes.
2. Object Detection on clean thali images.
3. BEV transformation for natural images and detection.
4. Bonus: NRF creation.

## Dataset Details
- Location: dataset-001/khana/ with subfolders per class.
- Labels: Khana Dataset/labels.txt (80 classes).
- Taxonomy: Khana Dataset/taxonomy.csv (categories, dietary info).

## Task Structure
Starting with Step 1: Image Classification.

# Extract Step 1 Requirements

## Task 1: Image Classification
- Design and train a classifier for 80 classes.
- Baseline validation accuracy: ~91%.
- Need to achieve better than baseline.
- Evaluation: 20-30 test images, leaderboard based.

## Requirements
- Load dataset from folders.
- Preprocess images.
- Use PyTorch/TensorFlow.
- Train model (e.g., ResNet).
- Validate accuracy >91%.
- Prepare for test evaluation.

# Implement Step 1

## Data Loading and Preprocessing

In [1]:
import torch
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm import tqdm
import os

In [2]:
# Set data path
data_path = 'dataset-001/khana/'

# Define transforms
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load dataset
dataset = ImageFolder(root=data_path, transform=transform)

# Split into train and val (80-20)
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])

# Data loaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

print(f'Classes: {dataset.classes}')
print(f'Num classes: {len(dataset.classes)}')
print(f'Train size: {len(train_dataset)}, Val size: {len(val_dataset)}')

Classes: ['aloo gobi', 'aloo methi', 'aloo mutter', 'aloo paratha', 'amritsari kulcha', 'anda curry', 'balushahi', 'banana chips', 'besan laddu', 'bhindi masala', 'biryani', 'boondi laddu', 'chaas', 'chana masala', 'chapati', 'chicken pizza', 'chicken wings', 'chikki', 'chivda', 'chole bhature', 'dabeli', 'dal khichdi', 'dhokla', 'falooda', 'fish curry', 'gajar ka halwa', 'garlic bread', 'garlic naan', 'ghevar', 'grilled sandwich', 'gujhia', 'gulab jamun', 'hara bhara kabab', 'idiyappam', 'idli', 'jalebi', 'kaju katli', 'khakhra', 'kheer', 'kulfi', 'margherita pizza', 'masala dosa', 'masala papad', 'medu vada', 'misal pav', 'modak', 'moong dal halwa', 'murukku', 'mysore pak', 'navratan korma', 'neer dosa', 'onion pakoda', 'palak paneer', 'paneer masala', 'paneer pizza', 'pani puri', 'paniyaram', 'papdi chaat', 'patrode', 'pav bhaji', 'pepperoni pizza', 'phirni', 'poha', 'pongal', 'puri bhaji', 'rajma chawal', 'rasgulla', 'rava dosa', 'sabudana khichdi', 'sabudana vada', 'samosa', 'seek

In [3]:
# Build a small CNN from scratch for an end-to-end pipeline
class SimpleKhanaCNN(torch.nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = torch.nn.Sequential(
            torch.nn.Conv2d(3, 32, kernel_size=3, padding=1),
            torch.nn.ReLU(inplace=True),
            torch.nn.BatchNorm2d(32),
            torch.nn.MaxPool2d(kernel_size=2),
            torch.nn.Conv2d(32, 64, kernel_size=3, padding=1),
            torch.nn.ReLU(inplace=True),
            torch.nn.BatchNorm2d(64),
            torch.nn.MaxPool2d(kernel_size=2),
            torch.nn.Conv2d(64, 128, kernel_size=3, padding=1),
            torch.nn.ReLU(inplace=True),
            torch.nn.BatchNorm2d(128),
            torch.nn.MaxPool2d(kernel_size=2),
            torch.nn.Dropout2d(0.2),
        )
        self.pool = torch.nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = torch.nn.Sequential(
            torch.nn.Flatten(),
            torch.nn.Linear(128, 512),
            torch.nn.ReLU(inplace=True),
            torch.nn.Dropout(0.5),
            torch.nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        x = self.classifier(x)
        return x

num_classes = len(dataset.classes)
model = SimpleKhanaCNN(num_classes)

# Move to device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

# Loss, optimizer, scheduler
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

print(f'Device: {device}')
print(f'Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad)}')

Device: cpu
Trainable parameters: 200784


In [4]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    for inputs, labels in tqdm(loader):
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    return running_loss / len(loader), 100. * correct / total

def validate_epoch(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    return running_loss / len(loader), 100. * correct / total

In [ ]:
# First run: small model, few epochs, end-to-end pipeline
num_epochs = 3
for epoch in range(num_epochs):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = validate_epoch(model, val_loader, criterion, device)
    scheduler.step()
    print(f'Epoch {epoch+1}/{num_epochs}: Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%, Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%')

  0%|          | 0/3296 [00:00<?, ?it/s]c:\Users\shubh\Downloads\MSc 4th Sem\CV ISM\Code\.venv\Lib\site-packages\PIL\Image.py:1034: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
  0%|          | 16/3296 [00:25<1:22:30,  1.51s/it]

# Validate Step 1 Output

## Check Validation Accuracy

In [ ]:
# Final validation accuracy
_, final_val_acc = validate_epoch(model, val_loader, criterion, device)
print(f'Final Validation Accuracy: {final_val_acc:.2f}%')
if final_val_acc > 91:
    print('Success: Above baseline!')
else:
    print('Below baseline, need improvement.')